# shared_utils Reference Notebook

Interactive examples showing common workflows with `shared_utils`.

For the full API reference, see [docs/SHARED_UTILS_API.md](../docs/SHARED_UTILS_API.md).

**Run the setup cell below first** to make `shared_utils` importable.

**Sections:**
1. [SimpleProcessor — High-Level Workflow](#1-simpleprocessor)
2. [quick_process — One-Liner](#2-quick_process)
3. [target_crs — Skip or Customize Reprojection](#3-target_crs)
4. [Local COG Conversion (No S3)](#4-local-cog-conversion)
5. [GeoTIFF Analysis](#5-geotiff-analysis)
6. [S3 Operations](#6-s3-operations)
7. [File Naming Helpers](#7-file-naming)
8. [Batch Analysis](#8-batch-analysis)
9. [Verification](#9-verification)

In [ ]:
# Setup: Add the repo root to sys.path so shared_utils is importable
# Run this cell first before any other cells
import sys
from pathlib import Path

repo_root = str(Path('..').resolve())
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"Added to path: {repo_root}")
print("shared_utils is now importable")

---
## 1. SimpleProcessor

The main high-level interface for processing disaster imagery from S3.

In [ ]:
from shared_utils.notebook_helpers import SimpleProcessor

config = {
    'event_name': '202510_Flood_AK',
    'bucket': 'nasa-disasters',
    'source_path': 'drcs_activations/202510_Flood_AK/sentinel2',
    'destination_base': 'drcs_activations_new',
    'target_crs': 'EPSG:4326',     # or None to keep original CRS
    'overwrite': False,
    'verify': True,

    # Optional: custom categorization patterns (regex)
    'categorization_patterns': {
        'trueColor': r'trueColor|truecolor|true_color|RGB',
        'colorInfrared': r'colorInfrared|colorIR|color_infrared',
        'NDVI': r'NDVI|ndvi',
    },

    # Optional: custom output directories per category
    'output_dirs': {
        'trueColor': 'Sentinel-2/trueColor',
        'colorInfrared': 'Sentinel-2/colorIR',
        'NDVI': 'Sentinel-2/NDVI',
    },

    # Optional: per-category nodata values
    'nodata_values': {
        'NDVI': -9999,
        'trueColor': None,  # auto-detect
    },
}

processor = SimpleProcessor(config)

In [ ]:
# Step 1: Connect to S3
processor.connect_to_s3()

# Step 2: Discover and categorize files
num_files = processor.discover_files()
print(f"Found {num_files} files")

# Step 3: Preview what will be processed
processor.preview_processing()

# Step 4: Process all files
results_df = processor.process_all()
results_df.head()

---
## 2. quick_process

One function that does everything above in a single call.

In [ ]:
from shared_utils.notebook_helpers import quick_process

results = quick_process({
    'event_name': '202510_Flood_AK',
    'bucket': 'nasa-disasters',
    'source_path': 'drcs_activations/202510_Flood_AK/sentinel2',
    'destination_base': 'drcs_activations_new',
    'overwrite': False,
})

---
## 3. target_crs

Control whether files are reprojected during COG conversion.

By default, all files are reprojected to **EPSG:4326**. Set `target_crs` to skip reprojection or use a different CRS.

In [ ]:
# Option 1: Reproject to EPSG:4326 (default)
config_reproject = {
    'event_name': '202510_Flood_AK',
    'bucket': 'nasa-disasters',
    'source_path': 'drcs_activations/202510_Flood_AK/sentinel2',
    'destination_base': 'drcs_activations_new',
    'target_crs': 'EPSG:4326',
}

# Option 2: Keep original CRS (no reprojection)
config_native = {
    'event_name': '202510_Flood_AK',
    'bucket': 'nasa-disasters',
    'source_path': 'drcs_activations/202510_Flood_AK/sentinel2',
    'destination_base': 'drcs_activations_new',
    'target_crs': None,        # Any of these work: None, 'None', 'none', ''
}

# Option 3: Reproject to a different CRS
config_utm = {
    'event_name': '202510_Flood_AK',
    'bucket': 'nasa-disasters',
    'source_path': 'drcs_activations/202510_Flood_AK/sentinel2',
    'destination_base': 'drcs_activations_new',
    'target_crs': 'EPSG:32606',  # UTM Zone 6N (Alaska)
}

The `target_crs` parameter works at every level:

```python
# High-level: SimpleProcessor / quick_process
config = {'target_crs': None, ...}

# Mid-level: main_processor.convert_to_cog
convert_to_cog(..., target_crs=None)

# Low-level: GDAL processor
create_cog_gdal(..., target_crs=None)
process_file_optimized(..., target_crs=None)

# Local conversion: cog_utils.convert_to_cog
convert_to_cog(input_tif, dst_crs=None)
```

---
## 4. Local COG Conversion

Convert local GeoTIFF files to COG without S3.

In [ ]:
from shared_utils.cog_utils import convert_to_cog, validate_cog, set_nodata_value

# Convert a local file to COG
output_path = convert_to_cog(
    input_tif='/path/to/input.tif',
    output_cog='/path/to/output_cog.tif',
    nodata=-9999,                # or None for auto-detect
    dst_crs='EPSG:4326',         # or None to keep original
    compression='ZSTD',
    compression_level=22,
    overview_levels=5,
    backend='rio',               # 'rio' (rio-cogeo) or 'gdal'
)
print(f"COG created: {output_path}")

# Validate the result
is_valid, details = validate_cog(output_path)
print(f"Valid COG: {is_valid}")
print(details)

In [ ]:
# Auto-select nodata based on data type
from shared_utils.cog_utils import set_nodata_value, validate_nodata_for_dtype

nodata = set_nodata_value('float32')     # Returns -9999.0
nodata = set_nodata_value('uint8')        # Returns 0
nodata = set_nodata_value('int16')        # Returns -9999

# Validate a nodata choice
result = validate_nodata_for_dtype(-9999, 'uint8')
print(result)  # {'valid': False, 'message': '...'}

---
## 5. GeoTIFF Analysis

Analyze files before processing to understand their properties.

In [ ]:
from shared_utils.geotiff_analyzer import analyze_geotiff, suggest_nodata_value, format_analysis_report

# Analyze a local file
results = analyze_geotiff('/path/to/file.tif')
print(format_analysis_report(results))

# Key fields in results:
# results['crs']          - Coordinate Reference System
# results['bounds']       - Geographic bounds
# results['shape']        - (bands, height, width)
# results['dtype']        - Data type
# results['nodata']       - Current nodata value
# results['compression']  - Current compression
# results['bands']        - Per-band statistics (min, max, mean, etc.)

In [ ]:
# Get nodata recommendation
suggestion = suggest_nodata_value(
    dtype=results['dtype'],
    bands=results['bands'],
    current_nodata=results['nodata']
)
print(f"Suggested nodata: {suggestion['suggested_value']}")
print(f"Reason: {suggestion['reason']}")

In [ ]:
# Analyze a file directly from S3
from shared_utils.geotiff_analyzer import analyze_s3_geotiff

results = analyze_s3_geotiff(
    bucket='nasa-disasters',
    key='drcs_activations/202510_Flood_AK/sentinel2/example.tif'
)

---
## 6. S3 Operations

Connect to S3, list files, upload/download.

In [ ]:
from shared_utils.s3_operations import (
    initialize_s3_client,
    list_s3_files,
    check_s3_file_exists,
    get_file_size_from_s3,
    download_from_s3,
    upload_to_s3,
)

# Initialize client (auto-detects credentials)
s3_client, fs_read = initialize_s3_client(bucket_name='nasa-disasters')

# List all TIF files under a prefix
files = list_s3_files(s3_client, 'nasa-disasters', 'drcs_activations/202510_Flood_AK', suffix='.tif')
print(f"Found {len(files)} files")
for f in files[:5]:
    size_gb = get_file_size_from_s3(s3_client, 'nasa-disasters', f)
    print(f"  {f} ({size_gb:.2f} GB)")

In [ ]:
# Check if a file exists
exists = check_s3_file_exists(s3_client, 'nasa-disasters', 'drcs_activations_new/some_file.tif')

# Download a file
download_from_s3(s3_client, 'nasa-disasters', 'drcs_activations/file.tif', '/tmp/file.tif')

# Upload a file
upload_to_s3(s3_client, '/tmp/output_cog.tif', 'nasa-disasters', 'drcs_activations_new/output_cog.tif')

In [ ]:
# Test upload permissions
from shared_utils.test_upload import test_s3_upload

can_upload = test_s3_upload(bucket='nasa-disasters')
print(f"Upload permissions: {'OK' if can_upload else 'DENIED'}")

---
## 7. File Naming

Parse filenames, extract dates, generate standardized names.

In [ ]:
from shared_utils.file_naming import (
    extract_date_from_filename,
    convert_date,
    parse_filename_components,
    create_cog_filename,
    create_output_path,
)

# Extract date from filename
date = extract_date_from_filename('S2_trueColor_20251015_alaska.tif')
print(f"Date: {date}")  # '20251015'

# Convert to formatted date
formatted = convert_date('20251015')
print(f"Formatted: {formatted}")  # '2025-10-15'

# Parse full filename
components = parse_filename_components('S2_trueColor_20251015_alaska.tif')
print(components)  # {'date': '20251015', 'satellite': ..., 'product': ..., ...}

# Generate standardized COG filename
cog_name = create_cog_filename('S2_trueColor_20251015_alaska.tif', '202510_Flood_AK')
print(f"COG name: {cog_name}")

# Build full output path
path = create_output_path('drcs_activations_new', 'Sentinel-2/trueColor', cog_name)
print(f"Output path: {path}")

---
## 8. Batch Analysis

Analyze multiple files in parallel and generate reports.

In [ ]:
from shared_utils.batch_analyzer import (
    analyze_batch_local,
    analyze_batch_s3,
    generate_summary_statistics,
    create_detailed_report,
)

# Analyze local files
import glob
files = glob.glob('/path/to/geotiffs/*.tif')
results = analyze_batch_local(files, max_workers=4)

# Or analyze from S3
results = analyze_batch_s3(
    bucket='nasa-disasters',
    prefix='drcs_activations/202510_Flood_AK/sentinel2',
    suffix='.tif',
    max_workers=4,
    limit=10  # analyze first 10 files only
)

# Summary statistics
summary = generate_summary_statistics(results)
print(f"Total files: {summary['total_files']}")
print(f"Data types: {summary['dtypes']}")
print(f"CRS values: {summary['crs_values']}")

# Detailed DataFrame report
df = create_detailed_report(results)
df.head()

---
## 9. Verification

Compare input and output files to verify processing integrity.

In [ ]:
from shared_utils.verification import compare_geotiffs, verify_s3_files

# Compare local files
comparison = compare_geotiffs(
    input_path='/path/to/original.tif',
    output_path='/path/to/cog_output.tif',
    band=1,
    sample_size=10000  # sample for large files
)
print(f"Data preserved: {comparison['data_preserved']}")
print(f"CRS match: {comparison['crs_match']}")
print(f"Nodata match: {comparison['nodata_match']}")

# Compare S3 files (downloads temporarily)
result = verify_s3_files(
    input_bucket='nasa-disasters',
    input_key='drcs_activations/original.tif',
    output_bucket='nasa-disasters',
    output_key='drcs_activations_new/output_cog.tif',
    save_dir='/tmp/verification'
)

---

## Module Quick Reference

| Level | Module | Primary Function | Use For |
|-------|--------|------------------|---------|
| High | `notebook_helpers` | `SimpleProcessor`, `quick_process` | Full S3 workflow |
| Mid | `main_processor` | `convert_to_cog` | Single S3 file conversion |
| Mid | `cog_processing` | `process_single_file` | Simplified S3 conversion |
| Low | `cog_utils` | `convert_to_cog` | Local file conversion |
| Low | `gdal_cog_processor` | `create_cog_gdal` | GDAL-native conversion |
| Tool | `geotiff_analyzer` | `analyze_geotiff` | Inspect file properties |
| Tool | `batch_analyzer` | `analyze_batch_s3` | Analyze many files |
| Tool | `verification` | `compare_geotiffs` | Verify processing quality |
| Tool | `file_naming` | `create_cog_filename` | Standardize filenames |
| Tool | `s3_operations` | `initialize_s3_client` | S3 connect/list/upload |
| Tool | `cog_validation` | `validate_cog` | Check COG validity |